> [!WARNING]
> **OFFLINE SUBMISSION NOTEBOOK**
> Internet is disabled. Upload the following as Kaggle datasets:
> 1. `proposed-cvga-checkpoints` — fold checkpoints (`fold0_best.pth` ... `fold4_best.pth`)
> 2. `timm-deps` — containing `timm-1.0.24-py3-none-any.whl`


# Proposed: DINOv3-ViT-L + Cross-View Gated Attention (CVGA)

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

Inference notebook for the **proposed CVGA model**. Uses DINOv3-ViT-L backbone with
Cross-View Gated Attention (CVGA) fusion blocks that perform bidirectional cross-attention
between left and right stereo image views. Includes metadata MLP fusion.

- **Backbone:** `vit_large_patch16_dinov3.lvd1689m` (DINOv3-ViT-L, 1024-d)
- **Fusion:** 2× CVGABlock (weight-tied bidirectional cross-view attention, AMP-safe)
- **Metadata:** 23 features (State/Species one-hot, NDVI, Height, month sin/cos)
- **5-fold ensemble** with compositional Softplus regression heads


## Inference and Submission Generation (CVGA)

In [ ]:
# --- Setup: install timm 1.0.24 (required for DINOv3 architecture) ---
import subprocess, sys, os
import glob as _glob

def _find(slug, pattern=None):
    """Find Kaggle dataset dir. If pattern given, searches subdirs too."""
    for base in [f'/kaggle/input/{slug}', *_glob.glob(f'/kaggle/input/datasets/*/{slug}')]:
        if not os.path.isdir(base): continue
        if pattern:
            if _glob.glob(os.path.join(base, pattern)): return base
            for sub in _glob.glob(os.path.join(base, '*')):
                if os.path.isdir(sub) and _glob.glob(os.path.join(sub, pattern)):
                    return sub
        return base
    raise FileNotFoundError(f"Dataset '{slug}' not found in /kaggle/input/")

subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    os.path.join(_find('timm-deps'), 'timm-1.0.24-py3-none-any.whl'),
    '--no-deps', '-q', '--force-reinstall'])
print("timm 1.0.24 installed")


In [ ]:
import os, gc, warnings
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

warnings.filterwarnings('ignore')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
class CFG:
    BASE_PATH = '/kaggle/input/competitions/csiro-biomass'
    TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
    TEST_IMAGE_DIR = os.path.join(BASE_PATH, 'test')
    MODEL_DIR = _find('proposed-cvga-checkpoints', '*.pth')
    MODEL_NAME = 'vit_large_patch16_dinov3.lvd1689m'
    N_FOLDS = 5
    FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
    IMG_SIZE = 512
    BATCH_SIZE = 5
    NUM_WORKERS = 0
    DROPOUT = 0.2
    USE_METADATA = True
    META_INPUT_DIM = 23
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {CFG.DEVICE}, Model: {CFG.MODEL_NAME}")
print(f"Checkpoints: {CFG.MODEL_DIR}")


In [ ]:
# --- Model Architecture (must match training exactly) ---
class CVGABlock(nn.Module):
    """Cross-View Gated Attention Block.
    Splits concatenated left+right tokens, performs bidirectional cross-attention
    between views, applies sigmoid gating, recombines with skip connection."""
    def __init__(self, d_model, n_heads=8, dropout=0.1, **kwargs):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm = nn.LayerNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.attn_drop_p = dropout

    def forward(self, x):
        B, S, D = x.shape; N = S // 2
        shortcut = x
        x = self.norm(x)
        g = torch.sigmoid(self.gate(x)); x = x * g
        left, right = x[:, :N], x[:, N:]

        def qkv(t):
            q = self.q_proj(t).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
            k = self.k_proj(t).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
            v = self.v_proj(t).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
            return q, k, v

        q_l, k_l, v_l = qkv(left)
        q_r, k_r, v_r = qkv(right)
        drop_p = self.attn_drop_p if self.training else 0.0
        left_out = F.scaled_dot_product_attention(q_l, k_r, v_r, dropout_p=drop_p)
        right_out = F.scaled_dot_product_attention(q_r, k_l, v_l, dropout_p=drop_p)
        left_out = left_out.transpose(1, 2).contiguous().view(B, N, D)
        right_out = right_out.transpose(1, 2).contiguous().view(B, N, D)
        x = torch.cat([left_out, right_out], dim=1)
        x = self.out_proj(x); x = self.drop(x)
        return shortcut + x

def _make_head(nf, dropout):
    return nn.Sequential(
        nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(nf//2, 1), nn.Softplus()
    )

class BiomassModelTimm(nn.Module):
    def __init__(self, model_name, dropout=0.2, use_cvga=True, pretrained=True,
                 use_metadata=False, meta_input_dim=23):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          num_classes=0, global_pool='')
        nf = self.backbone.num_features
        self.fusion = nn.Sequential(CVGABlock(nf, dropout=dropout), CVGABlock(nf, dropout=dropout))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.use_metadata = use_metadata
        if use_metadata:
            meta_hidden = 64
            self.meta_mlp = nn.Sequential(
                nn.Linear(meta_input_dim, meta_hidden), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(meta_hidden, meta_hidden),
            )
            self.meta_proj = nn.Sequential(nn.Linear(nf + meta_hidden, nf), nn.GELU())
        self.head_green = _make_head(nf, dropout)
        self.head_dead = _make_head(nf, dropout)
        self.head_clover = _make_head(nf, dropout)

    def forward(self, left, right, metadata=None):
        x_l = self.backbone(left); x_r = self.backbone(right)
        x = torch.cat([x_l, x_r], dim=1)
        x = self.fusion(x)
        x = self.pool(x.transpose(1, 2)).flatten(1)
        if self.use_metadata and metadata is not None:
            meta_feat = self.meta_mlp(metadata)
            x = self.meta_proj(torch.cat([x, meta_feat], dim=1))
        green = self.head_green(x); dead = self.head_dead(x); clover = self.head_clover(x)
        gdm = green + clover; total = gdm + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("CVGA model architecture defined")


In [ ]:
# --- Metadata Encoding ---
def encode_metadata(df):
    meta = df.copy()
    STATES = ['NSW', 'QLD', 'TAS', 'VIC']
    for s in STATES:
        meta[f'meta_state_{s}'] = (meta['State'] == s).astype(np.float32)
    SPECIES = [
        'Brachiaria decumbens', 'Chloris gayana', 'Digitaria eriantha',
        'Festuca arundinacea', 'Lolium multiflorum', 'Lolium perenne',
        'Megathyrsus maximus', 'Mixed', 'Paspalum dilatatum',
        'Pennisetum clandestinum', 'Setaria sphacelata',
        'Trifolium repens/Lolium perenne', 'Trifolium subterraneum',
        'Trifolium subterraneum/Lolium perenne',
        'Trifolium subterraneum/Phalaris aquatica'
    ]
    for sp in SPECIES:
        meta[f'meta_species_{sp}'] = (meta['Species'] == sp).astype(np.float32)
    meta['meta_ndvi'] = meta['Pre_GSHH_NDVI'].astype(np.float32)
    meta['meta_height'] = (meta['Height_Ave_cm'] / 100.0).astype(np.float32)
    month = pd.to_datetime(meta['Sampling_Date']).dt.month
    meta['meta_month_sin'] = np.sin(2 * np.pi * month / 12).astype(np.float32)
    meta['meta_month_cos'] = np.cos(2 * np.pi * month / 12).astype(np.float32)
    meta_cols = ([f'meta_state_{s}' for s in STATES] +
                 [f'meta_species_{sp}' for sp in SPECIES] +
                 ['meta_ndvi', 'meta_height', 'meta_month_sin', 'meta_month_cos'])
    meta[meta_cols] = meta[meta_cols].fillna(0.0)
    print(f"Metadata: {len(meta_cols)} features")
    return meta[meta_cols].values.astype(np.float32), len(meta_cols)


In [ ]:
# --- Test Data & Inference ---
def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

class TestBiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, meta_array=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values
        self.meta = meta_array
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img = cv2.imread(path)
        if img is None: img = np.zeros((1000,2000,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape; mid = w//2
        left, right = img[:, :mid], img[:, mid:]
        if self.transform:
            left = self.transform(image=left)['image']
            right = self.transform(image=right)['image']
        if self.meta is not None:
            return left, right, torch.tensor(self.meta[idx], dtype=torch.float32)
        return left, right

@torch.no_grad()
def predict_test(model, loader, device, use_meta=True):
    model.eval(); all_preds = []
    for batch in tqdm(loader, desc='Inference'):
        if use_meta:
            left, right, meta = batch
            meta = meta.to(device)
        else:
            left, right = batch
            meta = None
        with autocast('cuda'):
            preds = model(left.to(device), right.to(device), metadata=meta)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

test_long = pd.read_csv(CFG.TEST_CSV)
test_long['image_id'] = test_long['sample_id'].str.split('__').str[0]
test_df = test_long.drop_duplicates('image_id').reset_index(drop=True)
print(f"Test images: {len(test_df)}")

meta_array, meta_dim = encode_metadata(test_df)
print(f"Metadata shape: {meta_array.shape}")

test_dataset = TestBiomassDataset(test_df, CFG.TEST_IMAGE_DIR, get_val_transforms(), meta_array)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

all_fold_preds = []
for fold in CFG.FOLDS_TO_TRAIN:
    ckpt_path = os.path.join(CFG.MODEL_DIR, f'fold{fold}_best.pth')
    if not os.path.exists(ckpt_path):
        print(f'Checkpoint not found: {ckpt_path}, skipping fold {fold}.')
        continue
    model = BiomassModelTimm(
        CFG.MODEL_NAME, dropout=CFG.DROPOUT, use_cvga=True, pretrained=False,
        use_metadata=CFG.USE_METADATA, meta_input_dim=meta_dim
    ).to(CFG.DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=True))
    print(f'Loaded fold {fold}')
    all_fold_preds.append(predict_test(model, test_loader, CFG.DEVICE, use_meta=CFG.USE_METADATA))
    del model; gc.collect(); torch.cuda.empty_cache()

if not all_fold_preds:
    raise RuntimeError('No fold checkpoints found.')
avg_preds = np.mean(all_fold_preds, axis=0)
print(f'Ensemble of {len(all_fold_preds)} folds, shape: {avg_preds.shape}')


In [ ]:
# --- Submission ---
image_ids = test_df['image_id'].values
pred_map = {}
for i, img_id in enumerate(image_ids):
    for j, tn in enumerate(CFG.TARGET_COLS):
        pred_map[(img_id, tn)] = float(avg_preds[i, j])

test_long['target'] = test_long.apply(
    lambda row: pred_map.get((row['image_id'], row['target_name']), 0.0), axis=1)
df_sub = test_long[['sample_id','target']].copy()

sample_sub = pd.read_csv(os.path.join(CFG.BASE_PATH, 'sample_submission.csv'))
df_sub = sample_sub[['sample_id']].merge(df_sub, on='sample_id', how='left')
df_sub['target'] = df_sub['target'].fillna(0.0)
df_sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv, shape: {df_sub.shape}')
print(df_sub.head(10).to_string(index=False))
